### Initial Data Loading and Audit
We will load both datasets and check for:
- Row counts
- Date ranges
- Missing values and unusable columns

In [2]:
import polars as pl
import numpy as np

# Load datasets using Polars
rat_df = pl.read_csv('/content/rat_sightings copy.csv', ignore_errors=True)
inspections_df = pl.read_csv('/content/restaurant_inspections copy.csv', ignore_errors=True)

def audit_dataset_pl(df, name, date_col):
    print(f"--- Audit for {name} ---")
    print(f"Total Rows: {df.height}")

    # Attempt date conversion and check range
    try:
        temp_dates = df.select(pl.col(date_col).str.to_datetime(strict=False))
        min_d = temp_dates.select(pl.col(date_col).min()).item()
        max_d = temp_dates.select(pl.col(date_col).max()).item()
        print(f"Date Range: {min_d} to {max_d}")
    except Exception as e:
        print(f"Date error or column not found: {e}")

    # Identify unusable columns (all null or constant)
    unusable = []
    for col in df.columns:
        null_pct = df.select(pl.col(col).is_null().mean()).item()
        unique_count = df.select(pl.col(col).n_unique()).item()
        if null_pct > 0.95 or unique_count <= 1:
            unusable.append(col)

    print(f"Potentially Unusable Columns: {unusable}")
    print("\n")

audit_dataset_pl(rat_df, "Rat Sightings", "created_date")
audit_dataset_pl(inspections_df, "Restaurant Inspections", "INSPECTION DATE")

print("Data Previews:")
print(rat_df.head(2))
print(inspections_df.head(2))

--- Audit for Rat Sightings ---
Total Rows: 50954
Date Range: 2025-01-01 00:43:43 to 2026-09-17 01:35:48
Potentially Unusable Columns: ['complaint_type']


--- Audit for Restaurant Inspections ---
Total Rows: 10000
Date error or column not found: unable to find column "INSPECTION DATE"; valid columns: ["camis", "dba", "boro", "zipcode", "cuisine_description", "inspection_date", "violation_code", "violation_description", "critical_flag", "score", "grade"]
Potentially Unusable Columns: []


Data Previews:
shape: (2, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ unique_ke ┆ created_d ┆ closed_da ┆ status    ┆ … ┆ incident_ ┆ borough  ┆ latitude  ┆ longitude │
│ y         ┆ ate       ┆ te        ┆ ---       ┆   ┆ zip       ┆ ---      ┆ ---       ┆ ---       │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ str      ┆ f64       ┆ f64       │
│ i64       ┆ str       ┆ str       ┆           ┆   ┆ i64       ┆       

### Data Cleaning and Filtering
We will now filter for rodent-specific violations and prepare the data for merging, keeping in mind that we must count distinct restaurants (`camis`).

In [4]:
# Define broader rodent related keywords to ensure we catch violations
# Based on typical DOHMH violation descriptions
rodent_keywords = ["rat", "mice", "rodent", "mouse"]
rodent_pattern = "(?i)" + "|".join(rodent_keywords)

# 1. Clean Rat Sightings
rat_clean = rat_df.with_columns([
    pl.col("incident_zip").cast(pl.Utf8).alias("zip_code"),
    pl.col("created_date").str.to_datetime(strict=False)
]).select(["unique_key", "zip_code", "created_date", "latitude", "longitude"])

# 2. Clean Restaurant Inspections
inspections_clean = inspections_df.filter(
    pl.col("violation_description").str.contains(rodent_pattern)
).with_columns([
    pl.col("zipcode").cast(pl.Utf8).alias("zip_code"),
    pl.col("inspection_date").str.to_datetime(strict=False)
])

# Calculate metrics per ZIP: Distinct camis count
# 'Counting is not measuring' - we capture total count for context but focus on distinct camis
zip_metrics = inspections_clean.group_by("zip_code").agg([
    pl.col("camis").n_unique().alias("distinct_restaurants_with_rodents"),
    pl.len().alias("total_violations_in_zip")
])

print(f"Filtered Inspections Rows: {inspections_clean.height}")
print(f"Unique ZIPs with rodent violations: {zip_metrics.height}")
display(zip_metrics.sort("distinct_restaurants_with_rodents", descending=True).head(5))

Filtered Inspections Rows: 3713
Unique ZIPs with rodent violations: 185


zip_code,distinct_restaurants_with_rodents,total_violations_in_zip
str,u32,u32
"""11354""",74,82
"""10003""",69,75
"""10002""",66,76
"""10001""",64,70
"""11372""",64,70


### Dataset Integration
We will now merge the rat sightings data with the restaurant inspection metrics at the ZIP code level. To provide better context, we'll calculate the 'Rat Sightings per Restaurant' ratio to avoid the 'popularity contest' pitfall.

In [5]:
# 1. Aggregate Rat Sightings by ZIP
rat_zip_counts = rat_clean.group_by("zip_code").agg([
    pl.len().alias("total_rat_sightings")
])

# 2. Join the two datasets on zip_code
combined_df = zip_metrics.join(rat_zip_counts, on="zip_code", how="outer")

# 3. Create context-aware metrics
# We'll fill nulls with 0 for calculation
combined_df = combined_df.fill_null(0).with_columns([
    (pl.col("total_rat_sightings") / pl.col("distinct_restaurants_with_rodents").replace(0, 1)).alias("sightings_per_affected_restaurant")
])

print(f"Combined Dataset Size: {combined_df.height} rows (ZIP codes)")
display(combined_df.sort("sightings_per_affected_restaurant", descending=True).head(10))

Combined Dataset Size: 200 rows (ZIP codes)


/tmp/ipykernel_2502/158342449.py:7: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  combined_df = zip_metrics.join(rat_zip_counts, on="zip_code", how="outer")


zip_code,distinct_restaurants_with_rodents,total_violations_in_zip,zip_code_right,total_rat_sightings,sightings_per_affected_restaurant
str,u32,u32,str,u32,f64
null,0,0,"""10039""",326,326.0
"""10035""",7,8,"""10035""",1501,214.428571
"""10312""",2,2,"""10312""",265,132.5
"""10452""",9,11,"""10452""",1144,127.111111
"""10030""",5,7,"""10030""",489,97.8
"""10460""",4,5,"""10460""",361,90.25
"""11233""",15,16,"""11233""",1128,75.2
"""10308""",2,2,"""10308""",143,71.5
"""10026""",9,11,"""10026""",615,68.333333


### Interactive Geographic Analysis
We will create a Plotly map. We'll use a scatter map for the individual rat sightings and aggregate stats at the ZIP level to provide the context you requested.

In [25]:
import plotly.express as px
import polars as pl
from ipywidgets import widgets, VBox # Import VBox
from IPython.display import display, HTML # Import display and HTML explicitly
import plotly.io as pio # Import plotly.io for renderer settings

# Set default renderer for Colab
pio.renderers.default = "colab"

# 1. Prepare representative coordinates for each ZIP code
zip_coords = rat_clean.filter(pl.col("latitude").is_not_null()).group_by("zip_code").agg([
    pl.col("latitude").mean().alias("Latitude"),
    pl.col("longitude").mean().alias("Longitude")
])

# 2. Prepare final display data with human-readable labels for the UI
display_data = combined_df.join(zip_coords, on="zip_code", how="inner").select([
    pl.col("zip_code").alias("ZIP Code"),
    pl.col("distinct_restaurants_with_rodents").alias("Unique Restaurants with Violations"),
    pl.col("total_rat_sightings").alias("Total Rat Sightings"),
    pl.col("sightings_per_affected_restaurant").alias("Sightings per Restaurant"),
    pl.col("Latitude"),
    pl.col("Longitude")
])

pd_display = display_data.to_pandas()
zips = sorted(pd_display['ZIP Code'].unique().tolist())

def update_map(selected_zip):
    # print("DEBUG: update_map called with:", selected_zip) # DEBUG PRINT - Removed as it clutters if not displaying correctly
    filtered_df = pd_display
    if selected_zip != 'All':
        filtered_df = pd_display[pd_display['ZIP Code'] == selected_zip]

    # print(f"Rendering map for ZIP: {selected_zip}. Data points: {len(filtered_df)}") # DEBUG PRINT - Removed

    if filtered_df.empty:
        # print("No data to display for the selected ZIP code.") # DEBUG PRINT - Removed
        fig = px.scatter_mapbox(title="No data to display for selected ZIP code.")
        fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
        return fig

    # print("Filtered data head for map:") # DEBUG PRINT - Removed
    # print(filtered_df.head()) # DEBUG PRINT - Removed

    # Default center for NYC if 'All' is selected or no specific ZIP is provided
    map_center = {"lat": 40.75, "lon": -74.00} # Approx center of NYC: Empire State Building coordinates

    # Adjust zoom level for 'All' vs. specific ZIP
    map_zoom = 10 if selected_zip == 'All' else 13

    # If specific ZIP is selected and has data, center on that ZIP's coordinates
    if selected_zip != 'All' and not filtered_df.empty:
        map_center = {"lat": filtered_df['Latitude'].mean(), "lon": filtered_df['Longitude'].mean()}

    # Use open-street-map style to guarantee no API key is requested
    fig = px.scatter_mapbox(
        filtered_df,
        lat="Latitude",
        lon="Longitude",
        size="Sightings per Restaurant",
        color="ZIP Code",
        hover_name="ZIP Code",
        hover_data={
            "ZIP Code": False,
            "Latitude": False,
            "Longitude": False,
            "Total Rat Sightings": True,
            "Unique Restaurants with Violations": True,
            "Sightings per Restaurant": ":.2f"
        },
        zoom=map_zoom,
        center=map_center,
        mapbox_style="open-street-map",
        title=f"NYC Rodent Analysis: {'Citywide Overview' if selected_zip == 'All' else 'ZIP Code ' + selected_zip}",
        height=600
    )

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        legend_title_text="ZIP Code Identifier"
    )
    return fig

# Create the dropdown widget
zip_dropdown = widgets.Dropdown(
    options=['All'] + zips,
    value='All',
    description='Filter ZIP:'
)

# Create an Output widget to hold the map
map_output = widgets.Output()

# Define the display function, which will be executed when dropdown value changes
def display_map_content(selected_zip_val):
    print(f"DEBUG: display_map_content called for {selected_zip_val}") # DEBUG PRINT
    with map_output:
        map_output.clear_output(wait=True) # Clear previous output before drawing new map
        # Add a simple HTML heading to test if the Output widget displays content at all
        display(HTML(f"<h2>Loading map for ZIP: {selected_zip_val}...</h2>"))
        fig = update_map(selected_zip_val)
        # Explicitly display the figure as HTML fragment to ensure rendering in ipywidgets.Output
        # Setting full_html=False is crucial for embedding within another HTML context like an Output widget
        display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))
    print(f"DEBUG: display_map_content finished for {selected_zip_val}") # DEBUG PRINT

# Observe the dropdown for changes
zip_dropdown.observe(lambda change: display_map_content(change['new']), names='value')

# Display the widgets vertically using VBox
ui = VBox([zip_dropdown, map_output])
display(ui)

# Trigger initial display immediately after displaying the widgets
display_map_content(zip_dropdown.value)

DEBUG: display_map_content called for All
DEBUG: display_map_content finished for All
